# Intelligent Document Processing with Blueprints in Amazon Bedrock Data Automation

## Introduction

[Amazon Bedrock Data Automation (BDA)](https://docs.aws.amazon.com/bedrock/latest/userguide/bda.html) is a fully managed capability of Amazon Bedrock that streamlines the generation of valuable insights from unstructured, multimodal content such as documents, images, audio, and videos. With BDA, you can build automated **Intelligent Document Processing (IDP)**, media analysis, and Retrieval-Augmented Generation (RAG) workflows quickly and cost-effectively.

In this notebook, we demonstrate how to use BDA **Blueprints** to extract structured, custom insights from a scanned US bank check — including **reading handwritten notes** and **detecting whether the check is signed**. Blueprints let you define exactly which fields to extract from a document, going beyond generic OCR to produce domain-specific, structured output.

### What You Will Learn

By the end of this notebook you will understand how to:

1. **Set up** the environment and configure S3 locations for BDA input/output
2. **Prepare** a sample scanned bank check document and upload it to S3
3. **Create a custom Blueprint** that defines the fields to extract (memo, account number, routing number, check number, etc.)
4. **Invoke BDA** asynchronously to process the document using the custom blueprint
5. **Monitor** the invocation job status
6. **Retrieve and explore** the extracted custom insights, including confidence scores and explainability info

### Workflow Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────────┐
│  Upload scanned  │────▶│  Select custom    │────▶│  Select/Create BDA  │
│  bank check to   │     │  Blueprint based  │     │  Project with       │
│  S3 bucket       │     │  on use-case      │     │  Blueprint(s)       │
└─────────────────┘     └──────────────────┘     └─────────┬───────────┘
                                                           │
                                                           ▼
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────────┐
│  View custom     │◀────│  Retrieve job    │◀────│  InvokeDataAuto-    │
│  output with     │     │  metadata &      │     │  mationAsync API    │
│  Blueprint       │     │  results from S3 │     │  for IDP            │
└─────────────────┘     └──────────────────┘     └─────────────────────┘
```


## Prerequisites

Before running this notebook, ensure you have:

1. **AWS Account** with access to Amazon Bedrock (BDA must be enabled in your region)
2. **IAM Permissions** — The execution role needs the following managed policies (or equivalent):
   - `AmazonBedrockFullAccess`
   - `AmazonS3FullAccess`
   - `AmazonSageMakerFullAccess`
   - `IAMReadOnlyAccess`
3. **SageMaker Studio** notebook instance (or any Jupyter environment with AWS credentials configured)
4. **Sample document** — A scanned bank check image placed in `data/documents/` directory
5. **Blueprint schema** — A JSON schema file placed in `data/blueprints/` directory
6. **Utility modules** — `utils/helper_functions.py` and `utils/display_functions.py` must be available

> **Note:** Run the cells in order. If you need to restart, use *Kernel → Restart Kernel* from the menu.

## Table of Contents

| Step | Description |
|------|-------------|
| **Step 1** | [Install dependencies and import libraries](#step-1) |
| **Step 2** | [Configure S3 locations and initialize clients](#step-2) |
| **Step 3** | [Prepare and upload the sample document](#step-3) |
| **Step 4** | [Create or update the custom Blueprint](#step-4) |
| **Step 5** | [Invoke BDA asynchronously](#step-5) |
| **Step 6** | [Monitor job status](#step-6) |
| **Step 7** | [Retrieve and explore job metadata](#step-7) |
| **Step 8** | [View custom output and insights](#step-8) |
| **Step 9** | [Explore document insights with confidence scores](#step-9) |

<a id="step-1"></a>
## Step 1: Install Dependencies and Import Libraries

We start by installing the required Python packages. BDA requires `boto3 >= 1.37.6`. We also use:
- **`itables`** — for interactive table rendering in Jupyter
- **`PyPDF2`** — for PDF manipulation
- **`sagemaker`** — for convenient access to the default S3 bucket

In [ ]:
%pip install --no-warn-conflicts "boto3>=1.37.6" itables==2.2.4 PyPDF2==3.0.1 --upgrade -qq

In [ ]:
# Enable auto-reload so changes to utility modules are picked up automatically
%load_ext autoreload
%autoreload 2

<a id="step-2"></a>
## Step 2: Configure S3 Locations and Initialize Clients

BDA's `InvokeDataAutomationAsync` API uses Amazon S3 for both input files and output results. In this step we:

1. Import the necessary libraries
2. Create boto3 clients for BDA, BDA Runtime, S3, and STS
3. Define S3 input/output locations using the SageMaker default bucket

> **Tip:** The SageMaker default bucket follows the naming convention `sagemaker-{region}-{account_id}`.

In [ ]:
import boto3
import json
from IPython.display import JSON, IFrame, Markdown
import sagemaker
import pandas as pd
from utils import display_functions, helper_functions
from pathlib import Path
import os

# Initialize SageMaker session and get default bucket
session = sagemaker.Session()
default_bucket = session.default_bucket()
current_region = boto3.session.Session().region_name

# Get AWS account ID
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

# Initialize BDA and S3 clients
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
s3_client = boto3.client('s3')

# Define S3 locations for BDA input and output
bda_s3_input_location = f's3://{default_bucket}/bda/input'
bda_s3_output_location = f's3://{default_bucket}/bda/output'

print(f'Region:          {current_region}')
print(f'Account ID:      {account_id}')
print(f'Default Bucket:  {default_bucket}')
print(f'S3 Input:        {bda_s3_input_location}')
print(f'S3 Output:       {bda_s3_output_location}')

<a id="step-3"></a>
## Step 3: Prepare and Upload the Sample Document

In this step, we upload a scanned US bank check image to the S3 input location. BDA will process this document using our custom blueprint to extract structured fields.

The sample document is a **handwritten bank check** (`Sample-check.PNG`) stored locally in the `data/documents/` directory.

In [ ]:
# Define local and S3 paths for the sample document
local_download_path = 'data/documents'
local_file_name = 'Sample-check.PNG'
local_file_path = os.path.join(local_download_path, local_file_name)

# Construct the S3 URI and upload
document_s3_uri = f'{bda_s3_input_location}/{local_file_name}'
target_s3_bucket, target_s3_key = helper_functions.get_bucket_and_key(document_s3_uri)
s3_client.upload_file(local_file_path, target_s3_bucket, target_s3_key)

print(f'Local file:      {local_file_path}')
print(f'S3 key:          {target_s3_key}')
print(f'S3 URI:          {document_s3_uri}')

### View the Sample Document

Let's preview the scanned bank check that we will process with BDA.

In [ ]:
IFrame(local_file_path, width=800, height=500)

<a id="step-4"></a>
## Step 4: Create or Update the Custom Blueprint

### What is a Blueprint?

A **Blueprint** in BDA defines the schema for extracting structured data from documents. It specifies:
- The **fields** to extract (e.g., payee name, amount, date)
- The **data types** for each field
- **Instructions** that guide the extraction model

BDA includes several [sample blueprints](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-bp-sample.html) for common document types. In this notebook, we customize the `bedrock-data-automation-public-us-bank-check` catalog blueprint to extract additional details such as:

| Field | Description |
|-------|-------------|
| Memo | Whether a memo line is present and its value |
| Account Number | The bank account number |
| Routing Number | The bank routing/transit number |
| Check Number | The check serial number |
| Payee Name | The "Pay to the order of" recipient |
| Amount | The check amount |
| Date | The date on the check |
| Is Signed | Whether the check has been signed |

We use the `create_blueprint` API (or `update_blueprint` for existing blueprints). Each blueprint is an AWS resource with its own **Blueprint ID** and **ARN**.

> **Schema location:** `data/blueprints/check-blueprint.json`

In [ ]:
# Define the blueprint configuration
blueprints = [
    {
        'name': 'Check-blueprint',
        'description': 'Blueprint for US Bank Check',
        'type': 'DOCUMENT',
        'stage': 'LIVE',
        'schema_path': 'data/blueprints/check-blueprint.json'
    }
]

In [ ]:
# Create or update each blueprint
blueprint_arns = []
for blueprint in blueprints:
    with open(blueprint['schema_path']) as f:
        blueprint_schema = json.load(f)
        blueprint_arn = helper_functions.create_or_update_blueprint(
            bda_client,
            blueprint['name'],
            blueprint['description'],
            blueprint['type'],
            blueprint['stage'],
            blueprint_schema
        )
        blueprint_arns.append(blueprint_arn)

print(f'Blueprint ARN(s): {blueprint_arns}')

### Inspect the Blueprint Schema

Let's view the JSON schema that defines which fields BDA will extract from the bank check. Each property in the schema maps to a field on the document.

In [ ]:
# Display the blueprint schema
with open('data/blueprints/check-blueprint.json') as f:
    schema = json.load(f)
JSON(schema, root='Blueprint Schema', expanded=True)

### Configure the BDA Project

A **BDA Project** groups one or more blueprints together and defines the standard/custom output configuration. The project ARN is used when invoking data automation.

You have two options:

- **Option A — Use an existing project:** If you already have a BDA project with the desired blueprint(s), set `USE_EXISTING_PROJECT = True` and provide the project ARN.
- **Option B — Create a new project:** If you are starting from scratch, set `USE_EXISTING_PROJECT = False`. The notebook will create a new project and associate the custom blueprint from Step 4.

> **Note:** Update the configuration in the cell below based on your scenario.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Configuration: Choose your project setup
# ──────────────────────────────────────────────────────────────────────

USE_EXISTING_PROJECT = False  # Set to False to create a new project

# Option A: Provide your existing project ARN
# ⚠️ Replace this with your own project ARN
EXISTING_PROJECT_ARN = 'arn:aws:bedrock:<region>:<account-id>:data-automation-project/<project-id>'

In [ ]:
if USE_EXISTING_PROJECT:
    # ── Option A: Use an existing project ──
    project_arn = EXISTING_PROJECT_ARN
    print(f'✅ Using existing project')
    print(f'   Project ARN: {project_arn}')
else:
    # ── Option B: Create a new project ──
    print('Creating a new BDA project...')

    # Standard output configuration (required by the API)
    standard_output_configuration = {
        'document': {
            'extraction': {
                'granularity': {'types': ['DOCUMENT', 'PAGE']},
                'boundingBox': {'state': 'ENABLED'}
            },
            'generativeField': {'state': 'ENABLED'},
            'outputFormat': {
                'textFormat': {'types': ['MARKDOWN']},
                'additionalFileFormat': {'state': 'ENABLED'}
            }
        }
    }

    # Custom output configuration referencing the blueprint from Step 4
    custom_output_configuration = {
        'blueprints': [
            {
                'blueprintArn': blueprint_arns[0],
                'blueprintStage': 'LIVE'
            }
        ]
    }

    create_response = bda_client.create_data_automation_project(
        projectName='check-blueprint-project',
        projectDescription='BDA project for US bank check processing with custom blueprint',
        projectStage='LIVE',
        standardOutputConfiguration=standard_output_configuration,
        customOutputConfiguration=custom_output_configuration,
        overrideConfiguration={'document': {'splitter': {'state': 'ENABLED'}}}
    )
    project_arn = create_response['projectArn']
    print(f'✅ Created new project')
    print(f'   Project ARN: {project_arn}')

<a id="step-5"></a>
## Step 5: Invoke Data Automation

With the project configured and the document uploaded to S3, we now invoke BDA to process the bank check.

The `InvokeDataAutomationAsync` API:
1. Accepts the **S3 URI** of the input document
2. Specifies the **S3 output location** for results
3. References the **BDA project** (which contains our blueprint)
4. Uses a **data automation profile** for processing

BDA will scan the document, split it into segments (if multi-page), and match each segment against the blueprints in the project.

In [ ]:
# Invoke BDA asynchronously
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': 'LIVE'
    },
    dataAutomationProfileArn=f'arn:aws:bedrock:{current_region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

invocationArn = response['invocationArn']
print(f'✅ Invoked data automation job')
print(f'   Invocation ARN: {invocationArn}')

<a id="step-6"></a>
## Step 6: Monitor Job Status

The `GetDataAutomationStatus` API lets us track the progress of the invocation job. The status transitions through:

```
Created  ──▶  InProgress  ──▶  Success
                              (or ClientError / ServiceError)
```

We poll until the job completes (typically 15–60 seconds depending on document complexity).

In [ ]:
# Wait for the BDA job to complete
status_response = helper_functions.wait_for_completion(
    client=bda_client,
    get_status_function=bda_runtime_client.get_data_automation_status,
    status_kwargs={'invocationArn': invocationArn},
    completion_states=['Success'],
    error_states=['ClientError', 'ServiceError'],
    status_path_in_response='status',
    max_iterations=15,
    delay=30
)

if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
    print(f'✅ Job completed successfully')
    print(f'   Metadata location: {job_metadata_s3_location}')
else:
    raise Exception(
        f'❌ Invocation Job Error\n'
        f'   Error type: {status_response.get("error_type", "Unknown")}\n'
        f'   Error message: {status_response.get("error_message", "Unknown")}'
    )

<a id="step-7"></a>
## Step 7: Retrieve and Explore Job Metadata

The job metadata contains:
- **Job ID** and **status** — unique identifier and completion status
- **Semantic modality** — the detected document type (e.g., `DOCUMENT`, `IMAGE`)
- **Segment metadata** — for each identified document segment:
  - S3 paths for **standard output** and **custom output**
  - **Custom output status**: `MATCH` (blueprint matched) or `NO_MATCH`
  - **Page indices** from the original file

A `MATCH` status means BDA successfully matched the segment to one of the blueprints in the project and produced structured custom output. `NO_MATCH` means only standard output is available.

In [ ]:
# Retrieve and parse job metadata from S3
job_metadata = json.loads(helper_functions.read_s3_object(job_metadata_s3_location))

# Display job status summary
job_id = job_metadata['job_id']
pd.set_option('display.max_colwidth', None)

job_status = pd.DataFrame({
    'Property': ['Job ID', 'Job Status', 'Semantic Modality'],
    'Value': [job_metadata['job_id'], job_metadata['job_status'], job_metadata['semantic_modality']]
})
print('Job Summary:')
display(job_status.style.hide(axis='index'))

# Display segment metadata in tabular and JSON views
job_metadata_table = pd.DataFrame(job_metadata['output_metadata'][0]['segment_metadata']).fillna('')
job_metadata_table.index.name = 'Segment Index'
job_metadata_json = JSON(job_metadata, root='job_metadata', expanded=True)

display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)],
    ['Table View', 'Raw JSON'])

<a id="step-8"></a>
## Step 8: View Custom Output and Insights

### Understanding BDA Output Types

BDA produces two types of output for each document segment:

| Output Type | Description |
|-------------|-------------|
| **Standard Output** | Generic extraction including text, tables, key-value pairs, and document structure |
| **Custom Output** | Blueprint-specific structured extraction with the exact fields defined in your schema |

The custom output includes:
- **`matched_blueprint`** — which blueprint was matched to this segment
- **`inference_result`** — the extracted field values
- **`explainability_info`** — confidence scores and source evidence for each extracted field

### Retrieve Segment Outputs

In [ ]:
# Extract standard and custom outputs for each segment
asset_id = 0
segments_metadata = next(
    item['segment_metadata']
    for item in job_metadata['output_metadata']
    if item['asset_id'] == asset_id
)

standard_outputs = [
    json.loads(helper_functions.read_s3_object(seg.get('standard_output_path')))
    for seg in segments_metadata
]

custom_outputs = [
    json.loads(helper_functions.read_s3_object(seg.get('custom_output_path')))
    if seg.get('custom_output_status') == 'MATCH' else None
    for seg in segments_metadata
]

matched_count = sum(1 for c in custom_outputs if c is not None)
print(f'Total segments: {len(segments_metadata)}')
print(f'Matched blueprints: {matched_count}')
print(f'Unmatched segments: {len(segments_metadata) - matched_count}')

### View Custom Output Summary

The summary shows which blueprint was matched to each segment along with the extracted inference results.

In [ ]:
# Display custom output summary in table and JSON views
custom_outputs_json = JSON(custom_outputs, root='custom_outputs', expanded=False)
custom_outputs_table = pd.DataFrame(helper_functions.get_summaries(custom_outputs)).fillna('')

display_functions.display_multiple(
    [
        display_functions.get_view(custom_outputs_table.style.hide(axis='index')),
        display_functions.get_view(custom_outputs_json)
    ],
    ['Table View', 'Raw JSON'])

<a id="step-9"></a>
## Step 9: Explore Document Insights with Confidence Scores

### Confidence Scores and Human-in-the-Loop

BDA provides **confidence scores** for each extracted field, indicating how certain the model is about the extraction. This is critical for production workflows where you may want to:

- ✅ **Auto-approve** fields with high confidence (e.g., ≥ 80%)
- ⚠️ **Flag for human review** fields with low confidence (e.g., < 80%)

The visualization below shows:
- Whether the **check is signed** (detected from the `is_signed` field)
- The **original document image** (left panel)
- The **extracted fields** with confidence scores (right panel)
- Fields below the 80% confidence threshold are highlighted for human verification

In [ ]:
# Visualize document insights with confidence scores
views = []
titles = []

for custom_output, standard_output in zip(custom_outputs, standard_outputs):
    if custom_output:
        inference_result = custom_output['inference_result']

        # ── Signature Detection ──
        is_signed = inference_result.get('is_signed', False)
        if is_signed:
            display(Markdown('### ✍️ Is the check signed? &nbsp; <span style="color:green;font-weight:bold;">YES</span>'))
        else:
            display(Markdown('### ✍️ Is the check signed? &nbsp; <span style="color:red;font-weight:bold;">NO</span>'))

        # Transform custom output with explainability info
        result = helper_functions.transform_custom_output(
            inference_result,
            custom_output['explainability_info'][0]
        )

        # Get document page images from standard output
        document_image_uris = [
            page.get('asset_metadata', {}).get('rectified_image')
            for page in standard_output.get('pages', [])
        ]

        # Build the segment view
        views.append(
            display_functions.segment_view(
                document_image_uris=document_image_uris,
                inference_result=result
            )
        )
        titles.append(
            custom_output.get('matched_blueprint', {}).get('name', 'Unknown Blueprint')
        )

# Display all segment views in tabs
display_functions.display_multiple(views, titles)

## Conclusion

In this notebook, we demonstrated how to use **Amazon Bedrock Data Automation (BDA)** with custom **Blueprints** to extract structured insights from a scanned bank check. Here's what we accomplished:

- **Created a custom Blueprint** with a JSON schema defining the specific fields to extract from a US bank check
- **Processed the document** using the `InvokeDataAutomationAsync` API with our custom blueprint
- **Retrieved structured output** including field values, matched blueprints, and confidence scores
- **Visualized the results** with side-by-side document images and extracted data, highlighting fields that may need human review

### Next Steps

- **Multi-document processing:** Combine multiple blueprints in a single project to process document packages (see the [Mortgage & Lending notebook](../21-Mortgage-and-Lending/21_mortgage_and_lending.ipynb))
- **Agent integration:** Use BDA with Amazon Bedrock Agents for end-to-end automated workflows (see the [Medical Claims Processing notebook](../22-Medical-Claims-Processing/22_medical_claims_processing.ipynb))
- **Custom blueprints:** Create blueprints for your own document types using the [Blueprint documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-bp.html)
- **Production deployment:** Integrate BDA into event-driven architectures using S3 events, EventBridge, and Lambda